In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import random
from pathlib import Path

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.applications import VGG16, ResNet50, EfficientNetB0, MobileNetV2
from tensorflow.keras.optimizers import Adam

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

print(f"TensorFlow version: {tf.__version__}")
print(f"GPU Available: {tf.config.list_physical_devices('GPU')}")


In [ ]:
# Set random seeds for reproducibility
np.random.seed(42)
tf.random.set_seed(42)
random.seed(42)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Paths - MODIFY THESE ACCORDING TO YOUR DATASET LOCATION
TRAIN_DIR = '/content/drive/MyDrive/collab_DL/CNN/clouds_train'
TEST_DIR = '/content/drive/MyDrive/collab_DL/CNN/clouds_test'

In [ ]:
# Image parameters
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
CHANNELS = 3

# Training parameters
EPOCHS = 100
LEARNING_RATE = 0.001
VALIDATION_SPLIT = 0.2  # We'll split train data into train/val

In [ ]:
# Class names
CLASS_NAMES = [
    'cirriform clouds',
    'clear sky',
    'cumulonimbus clouds',
    'cumulus clouds',
    'high cumuliform clouds',
    'stratiform clouds',
    'stratocumulus clouds'
]

NUM_CLASSES = len(CLASS_NAMES)
print(f"\nConfiguration:")
print(f"Image size: {IMG_HEIGHT}x{IMG_WIDTH}")
print(f"Batch size: {BATCH_SIZE}")
print(f"Number of classes: {NUM_CLASSES}")
print(f"Validation split: {VALIDATION_SPLIT}")


In [ ]:
def explore_dataset(data_dir, dataset_name="Dataset"):
    """Explore the dataset structure and count images"""
    print(f"\n{'='*60}")
    print(f"{dataset_name} Exploration")
    print(f"{'='*60}")

    class_counts = {}
    total_images = 0

    for class_name in CLASS_NAMES:
        class_path = os.path.join(data_dir, class_name)
        if os.path.exists(class_path):
            images = [f for f in os.listdir(class_path)
                     if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
            count = len(images)
            class_counts[class_name] = count
            total_images += count
            print(f"{class_name:25s}: {count:4d} images")

    print(f"{'Total':25s}: {total_images:4d} images")
    print(f"{'='*60}\n")

    return class_counts

# Explore train and test datasets
train_counts = explore_dataset(TRAIN_DIR, "TRAIN Dataset")
test_counts = explore_dataset(TEST_DIR, "TEST Dataset")

In [ ]:
def plot_class_distribution(train_counts, test_counts):
    """Plot the distribution of classes in train and test sets"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Train distribution
    classes = list(train_counts.keys())
    train_values = list(train_counts.values())

    ax1.bar(range(len(classes)), train_values, color='skyblue', edgecolor='navy')
    ax1.set_xlabel('Cloud Classes', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
    ax1.set_title('Train Dataset - Class Distribution', fontsize=14, fontweight='bold')
    ax1.set_xticks(range(len(classes)))
    ax1.set_xticklabels(classes, rotation=45, ha='right')
    ax1.grid(axis='y', alpha=0.3)

    for i, v in enumerate(train_values):
        ax1.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

    # Test distribution
    test_values = list(test_counts.values())

    ax2.bar(range(len(classes)), test_values, color='lightcoral', edgecolor='darkred')
    ax2.set_xlabel('Cloud Classes', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Number of Images', fontsize=12, fontweight='bold')
    ax2.set_title('Test Dataset - Class Distribution', fontsize=14, fontweight='bold')
    ax2.set_xticks(range(len(classes)))
    ax2.set_xticklabels(classes, rotation=45, ha='right')
    ax2.grid(axis='y', alpha=0.3)

    for i, v in enumerate(test_values):
        ax2.text(i, v + 1, str(v), ha='center', va='bottom', fontweight='bold')

    plt.tight_layout()
    plt.savefig('./class_distribution.png', dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
plot_class_distribution(train_counts, test_counts)

In [ ]:
def visualize_samples(data_dir, samples_per_class=3):
    """Visualize sample images from each class"""
    fig, axes = plt.subplots(NUM_CLASSES, samples_per_class,
                             figsize=(15, 3*NUM_CLASSES))

    for i, class_name in enumerate(CLASS_NAMES):
        class_path = os.path.join(data_dir, class_name)
        images = [f for f in os.listdir(class_path)
                 if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

        # Randomly select samples
        selected_images = random.sample(images, min(samples_per_class, len(images)))

        for j, img_name in enumerate(selected_images):
            img_path = os.path.join(class_path, img_name)
            img = plt.imread(img_path)

            axes[i, j].imshow(img)
            axes[i, j].axis('off')
            if j == 0:
                axes[i, j].set_title(f"{class_name}",
                                    fontsize=10, fontweight='bold', loc='left')

    plt.suptitle('Sample Images from Each Cloud Class',
                 fontsize=16, fontweight='bold', y=0.995)
    plt.tight_layout()
    plt.savefig('./sample_images.png', dpi=300, bbox_inches='tight')
    plt.show()


In [ ]:
visualize_samples(TRAIN_DIR, samples_per_class=3)

In [ ]:
# Data Augmentation for training
train_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True,
    zoom_range=0.2,
    shear_range=0.1,
    fill_mode='nearest'
)

# Only rescaling for validation and test
val_test_datagen = ImageDataGenerator(
    rescale=1./255,
    validation_split=VALIDATION_SPLIT
)

test_datagen = ImageDataGenerator(rescale=1./255)

# Create generators
train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=42
)

validation_generator = val_test_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=42
)

test_generator = test_datagen.flow_from_directory(
    TEST_DIR,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    shuffle=False
)

print(f"\nData Generators Created:")
print(f"Train samples: {train_generator.samples}")
print(f"Validation samples: {validation_generator.samples}")
print(f"Test samples: {test_generator.samples}")
print(f"\nClass indices: {train_generator.class_indices}")


In [ ]:
def visualize_augmentation(generator, num_images=5):
    """Visualize the effect of data augmentation"""
    # Get a batch
    images, labels = next(generator)

    fig, axes = plt.subplots(1, num_images, figsize=(15, 3))

    for i in range(num_images):
        axes[i].imshow(images[i])
        class_idx = np.argmax(labels[i])
        axes[i].set_title(f"{CLASS_NAMES[class_idx]}", fontsize=10)
        axes[i].axis('off')

    plt.suptitle('Augmented Training Images', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig('./augmented_samples.png', dpi=300, bbox_inches='tight')
    plt.show()

visualize_augmentation(train_generator)

## Simple CNN


In [ ]:
def create_simple_cnn():
    """Create a simple CNN model"""
    model = models.Sequential([
        # First Conv Block
        layers.Conv2D(32, (3, 3), activation='relu',
                     input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
        layers.MaxPooling2D((2, 2)),

        # Second Conv Block
        layers.Conv2D(64, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Third Conv Block
        layers.Conv2D(128, (3, 3), activation='relu'),
        layers.MaxPooling2D((2, 2)),

        # Flatten and Dense layers
        layers.Flatten(),
        layers.Dense(128, activation='relu'),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='Simple_CNN')

    return model

model_simple = create_simple_cnn()
model_simple.summary()


In [ ]:
model_simple.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model compiled successfully!")


In [ ]:
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True,
    verbose=1
)

reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-7,
    verbose=1
)

checkpoint_simple = ModelCheckpoint(
    './model_simple_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
callbacks = [early_stopping, reduce_lr, checkpoint_simple]

print("✓ Callbacks configured!")

In [ ]:
print("TRAINING SIMPLE CNN MODEL")
print("="*60 + "\n")

history_simple = model_simple.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks,
    verbose=1
)

print("\n✓ Training completed!")

In [ ]:
def plot_training_history(history, model_name="Model"):
    """Plot training and validation metrics"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

    # Accuracy
    ax1.plot(history.history['accuracy'], label='Train Accuracy', linewidth=2)
    ax1.plot(history.history['val_accuracy'], label='Val Accuracy', linewidth=2)
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title(f'{model_name} - Accuracy', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss
    ax2.plot(history.history['loss'], label='Train Loss', linewidth=2)
    ax2.plot(history.history['val_loss'], label='Val Loss', linewidth=2)
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax2.set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'./{model_name.lower().replace(" ", "_")}_history.png',
                dpi=300, bbox_inches='tight')
    plt.show()



In [ ]:
plot_training_history(history_simple, "Simple CNN")

In [ ]:
print("EVALUATION - SIMPLE CNN MODEL")
print("="*60 + "\n")

test_loss, test_accuracy = model_simple.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy*100:.2f}%")
print(f"Test Loss: {test_loss:.4f}")



## model 2

In [ ]:
def create_improved_cnn():
    """Create an improved CNN with Batch Normalization and more layers"""
    model = models.Sequential([
        # First Conv Block
        layers.Conv2D(32, (3, 3), padding='same',
                     input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(32, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Second Conv Block
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(64, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Third Conv Block
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(128, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Fourth Conv Block
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.Conv2D(256, (3, 3), padding='same'),
        layers.BatchNormalization(),
        layers.Activation('relu'),
        layers.MaxPooling2D((2, 2)),
        layers.Dropout(0.25),

        # Dense layers
        layers.Flatten(),
        layers.Dense(512, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(256, activation='relu'),
        layers.BatchNormalization(),
        layers.Dropout(0.5),
        layers.Dense(NUM_CLASSES, activation='softmax')
    ], name='Improved_CNN')

    return model

model_improved = create_improved_cnn()
model_improved.summary()


In [ ]:
model_improved.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_improved = ModelCheckpoint(
    './model_improved_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)
callbacks_improved = [early_stopping, reduce_lr, checkpoint_improved]


In [ ]:
print("TRAINING IMPROVED CNN MODEL")
print("="*60 + "\n")

history_improved = model_improved.fit(
    train_generator,
    epochs=EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks_improved,
    verbose=1
)

print("\n✓ Training completed!")

In [ ]:
plot_training_history(history_improved, "Improved CNN")

In [ ]:
print("EVALUATION - IMPROVED CNN MODEL")
print("="*60 + "\n")

test_loss_improved, test_accuracy_improved = model_improved.evaluate(test_generator)
print(f"\nTest Accuracy: {test_accuracy_improved*100:.2f}%")
print(f"Test Loss: {test_loss_improved:.4f}")

## model 3 - Transfer learning

In [ ]:
def plot_two_phase_training(history1, history2, model_name="Model"):
    """Plot training history for both phases"""
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    # Combine histories
    acc1 = history1.history['accuracy']
    val_acc1 = history1.history['val_accuracy']
    acc2 = history2.history['accuracy']
    val_acc2 = history2.history['val_accuracy']

    loss1 = history1.history['loss']
    val_loss1 = history1.history['val_loss']
    loss2 = history2.history['loss']
    val_loss2 = history2.history['val_loss']

    epochs1 = range(1, len(acc1) + 1)
    epochs2 = range(len(acc1) + 1, len(acc1) + len(acc2) + 1)

    # Accuracy plot
    ax1.plot(epochs1, acc1, 'b-', label='Phase 1 Train', linewidth=2)
    ax1.plot(epochs1, val_acc1, 'b--', label='Phase 1 Val', linewidth=2)
    ax1.plot(epochs2, acc2, 'r-', label='Phase 2 Train (Fine-tune)', linewidth=2)
    ax1.plot(epochs2, val_acc2, 'r--', label='Phase 2 Val (Fine-tune)', linewidth=2)
    ax1.axvline(x=len(acc1), color='green', linestyle=':', linewidth=2, label='Fine-tuning starts')
    ax1.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Accuracy', fontsize=12, fontweight='bold')
    ax1.set_title(f'{model_name} - Accuracy (Two-Phase Training)', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    # Loss plot
    ax2.plot(epochs1, loss1, 'b-', label='Phase 1 Train', linewidth=2)
    ax2.plot(epochs1, val_loss1, 'b--', label='Phase 1 Val', linewidth=2)
    ax2.plot(epochs2, loss2, 'r-', label='Phase 2 Train (Fine-tune)', linewidth=2)
    ax2.plot(epochs2, val_loss2, 'r--', label='Phase 2 Val (Fine-tune)', linewidth=2)
    ax2.axvline(x=len(acc1), color='green', linestyle=':', linewidth=2, label='Fine-tuning starts')
    ax2.set_xlabel('Epoch', fontsize=12, fontweight='bold')
    ax2.set_ylabel('Loss', fontsize=12, fontweight='bold')
    ax2.set_title(f'{model_name} - Loss (Two-Phase Training)', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(f'./{model_name.lower().replace(" ", "_")}_two_phase_history.png',
                dpi=300, bbox_inches='tight')
    plt.show()

### base model VGG16

In [ ]:
# Create base model
base_model_vgg16 = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)
)

# 🔒 PHASE 1: FREEZE ALL BASE MODEL LAYERS
print("Phase 1: Freezing all VGG16 base layers...")
base_model_vgg16.trainable = False

# Count frozen layers
frozen_layers = sum([not layer.trainable for layer in base_model_vgg16.layers])
print(f"✓ Frozen layers: {frozen_layers}/{len(base_model_vgg16.layers)}")

# Build model with custom top
model_vgg16 = models.Sequential([
    base_model_vgg16,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='VGG16_Phase1')

model_vgg16.summary()

In [ ]:
model_vgg16.compile(
    optimizer=Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)
print("\n✓ Model compiled for Phase 1 (training top layers only)")


In [ ]:
checkpoint_vgg16_phase1 = ModelCheckpoint(
    './model_vgg16_phase1_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_phase1 = [early_stopping, reduce_lr, checkpoint_vgg16_phase1]

# Train Phase 1 (typically fewer epochs needed)
PHASE1_EPOCHS = 15

history_vgg16_phase1 = model_vgg16.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=validation_generator,
    callbacks=callbacks_phase1,
    verbose=1
)

print("\n✓ Phase 1 training completed!")



In [ ]:
plot_training_history(history_vgg16_phase1, "VGG16 Phase 1")

In [ ]:
print("PHASE 2: FINE-TUNING (Unfreezing Top Layers)")
print("="*80 + "\n")

# 🔓 UNFREEZE the base model
base_model_vgg16.trainable = True

# Fine-tune from this layer onwards
# VGG16 has 19 layers, we'll unfreeze the last 4 layers (last conv block)
fine_tune_at = 15  # Freeze first 15 layers, unfreeze last 4

print(f"Freezing first {fine_tune_at} layers...")
for layer in base_model_vgg16.layers[:fine_tune_at]:
    layer.trainable = False

# Count trainable parameters
trainable_layers = sum([layer.trainable for layer in base_model_vgg16.layers])
print(f"✓ Trainable layers in base model: {trainable_layers}/{len(base_model_vgg16.layers)}")
print(f"✓ Fine-tuning last {trainable_layers} layers of VGG16")

# Re-compile with LOWER learning rate (crucial for fine-tuning!)
model_vgg16.compile(
    optimizer=Adam(learning_rate=1e-5),  # 100x smaller LR to not destroy pre-trained weights
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\n✓ Model re-compiled with lower learning rate (1e-5) for fine-tuning")


In [ ]:
# Setup callbacks for Phase 2
checkpoint_vgg16_phase2 = ModelCheckpoint(
    '/home/claude/model_vgg16_phase2_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

callbacks_phase2 = [early_stopping, reduce_lr, checkpoint_vgg16_phase2]


In [ ]:
# Train Phase 2
PHASE2_EPOCHS = 50
total_epochs = PHASE1_EPOCHS + PHASE2_EPOCHS

print("\n" + "="*80)
print("STARTING FINE-TUNING...")
print("="*80 + "\n")

history_vgg16_phase2 = model_vgg16.fit(
    train_generator,
    epochs=total_epochs,
    initial_epoch=len(history_vgg16_phase1.history['loss']),  # Continue from Phase 1
    validation_data=validation_generator,
    callbacks=callbacks_phase2,
    verbose=1
)

print("\n✓ Phase 2 fine-tuning completed!")

In [ ]:
plot_two_phase_training(history_vgg16_phase1, history_vgg16_phase2, "VGG16 Two-Phase")

In [ ]:
print("FINAL EVALUATION - VGG16 TWO-PHASE TRANSFER LEARNING")
print("="*80 + "\n")

test_loss_vgg16, test_accuracy_vgg16 = model_vgg16.evaluate(test_generator)
print(f"\nFinal Test Accuracy: {test_accuracy_vgg16*100:.2f}%")
print(f"Final Test Loss: {test_loss_vgg16:.4f}")

# Compare with Phase 1 only
print("\n" + "-"*80)
print("COMPARISON: Phase 1 vs Phase 2")
print("-"*80)
print(f"Phase 1 (top layers only) - Val Accuracy: {max(history_vgg16_phase1.history['val_accuracy'])*100:.2f}%")
print(f"Phase 2 (fine-tuned)      - Val Accuracy: {max(history_vgg16_phase2.history['val_accuracy'])*100:.2f}%")
print(f"Improvement: {(max(history_vgg16_phase2.history['val_accuracy']) - max(history_vgg16_phase1.history['val_accuracy']))*100:.2f}%")
print("-"*80)


### base model EfficientNetB0

In [ ]:
# Create base model
base_model_effnet = EfficientNetB0(
    weights='imagenet',
    include_top=False,
    input_shape=(IMG_HEIGHT, IMG_WIDTH, CHANNELS)
)

# PHASE 1: Freeze all
base_model_effnet.trainable = False

model_effnet = models.Sequential([
    base_model_effnet,
    layers.GlobalAveragePooling2D(),
    layers.Dense(512, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(256, activation='relu'),
    layers.BatchNormalization(),
    layers.Dropout(0.5),
    layers.Dense(NUM_CLASSES, activation='softmax')
], name='EfficientNetB0_Phase1')

model_effnet.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
print("Phase 1: Training top layers only...")

checkpoint_effnet_phase1 = ModelCheckpoint(
    './model_efficientnet_phase1_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

history_effnet_phase1 = model_effnet.fit(
    train_generator,
    epochs=PHASE1_EPOCHS,
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr, checkpoint_effnet_phase1],
    verbose=1
)

print("\n✓ EfficientNet Phase 1 completed!")

In [ ]:
print("EfficientNetB0 - PHASE 2: FINE-TUNING")

# Unfreeze base model
base_model_effnet.trainable = True

# EfficientNet has ~237 layers, we'll fine-tune the last ~20%
fine_tune_at = int(len(base_model_effnet.layers) * 0.8)  # Freeze 80%, train 20%

for layer in base_model_effnet.layers[:fine_tune_at]:
    layer.trainable = False

trainable_layers = sum([layer.trainable for layer in base_model_effnet.layers])
print(f"✓ Fine-tuning last {trainable_layers} layers (out of {len(base_model_effnet.layers)})")

# Re-compile with lower LR
model_effnet.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)


In [ ]:
checkpoint_effnet_phase2 = ModelCheckpoint(
    './model_efficientnet_phase2_best.keras',
    monitor='val_accuracy',
    save_best_only=True,
    verbose=1
)

print("\nStarting fine-tuning...")

history_effnet_phase2 = model_effnet.fit(
    train_generator,
    epochs=total_epochs,
    initial_epoch=len(history_effnet_phase1.history['loss']),
    validation_data=validation_generator,
    callbacks=[early_stopping, reduce_lr, checkpoint_effnet_phase2],
    verbose=1
)

print("\n✓ EfficientNet Phase 2 completed!")


In [ ]:
plot_two_phase_training(history_effnet_phase1, history_effnet_phase2, "EfficientNetB0 Two-Phase")

In [ ]:
print("\n" + "="*80)
print("FINAL EVALUATION - EfficientNetB0 TWO-PHASE TRANSFER LEARNING")
print("="*80 + "\n")

test_loss_effnet, test_accuracy_effnet = model_effnet.evaluate(test_generator)
print(f"\nFinal Test Accuracy: {test_accuracy_effnet*100:.2f}%")
print(f"Final Test Loss: {test_loss_effnet:.4f}")

print("\n" + "-"*80)
print("COMPARISON: Phase 1 vs Phase 2")
print("-"*80)
print(f"Phase 1 (top layers only) - Val Accuracy: {max(history_effnet_phase1.history['val_accuracy'])*100:.2f}%")
print(f"Phase 2 (fine-tuned)      - Val Accuracy: {max(history_effnet_phase2.history['val_accuracy'])*100:.2f}%")
print(f"Improvement: {(max(history_effnet_phase2.history['val_accuracy']) - max(history_effnet_phase1.history['val_accuracy']))*100:.2f}%")
print("-"*80)

### Compare models

In [ ]:
def compare_models():
    """Compare all models performance"""
    models_comparison = {
        'Simple CNN': test_accuracy,
        'Improved CNN': test_accuracy_improved,
        'VGG16 Transfer': test_accuracy_vgg16,
        'EfficientNetB0 Transfer': test_accuracy_effnet
    }

    # Create comparison plot
    fig, ax = plt.subplots(figsize=(12, 6))

    models = list(models_comparison.keys())
    accuracies = [acc * 100 for acc in models_comparison.values()]
    colors = ['#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A']

    bars = ax.bar(models, accuracies, color=colors, edgecolor='black', linewidth=1.5)

    ax.set_ylabel('Test Accuracy (%)', fontsize=14, fontweight='bold')
    ax.set_title('Model Performance Comparison', fontsize=16, fontweight='bold')
    ax.set_ylim([0, 100])
    ax.grid(axis='y', alpha=0.3)

    # Add value labels on bars
    for bar, acc in zip(bars, accuracies):
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2., height,
                f'{acc:.2f}%',
                ha='center', va='bottom', fontsize=12, fontweight='bold')

    plt.xticks(rotation=15, ha='right')
    plt.tight_layout()
    plt.savefig('./models_comparison.png', dpi=300, bbox_inches='tight')
    plt.show()

    # Print comparison table
    print("\n" + "="*60)
    print("MODELS PERFORMANCE COMPARISON")
    print("="*60)
    print(f"{'Model':<30} {'Test Accuracy':<15} {'Test Loss':<15}")
    print("-"*60)
    print(f"{'Simple CNN':<30} {test_accuracy*100:>13.2f}% {test_loss:>14.4f}")
    print(f"{'Improved CNN':<30} {test_accuracy_improved*100:>13.2f}% {test_loss_improved:>14.4f}")
    print(f"{'VGG16 Transfer':<30} {test_accuracy_vgg16*100:>13.2f}% {test_loss_vgg16:>14.4f}")
    print(f"{'EfficientNetB0 Transfer':<30} {test_accuracy_effnet*100:>13.2f}% {test_loss_effnet:>14.4f}")
    print("="*60)

    # Find best model
    best_model_name = max(models_comparison, key=models_comparison.get)
    best_accuracy = models_comparison[best_model_name] * 100
    print(f"\n🏆 BEST MODEL: {best_model_name} with {best_accuracy:.2f}% accuracy")

    return models_comparison

In [ ]:
models_comparison = compare_models()

## Consusion matrix

In [ ]:
def plot_confusion_matrix(model, generator, model_name="Model"):
    """Plot confusion matrix for the model"""
    # Get predictions
    generator.reset()
    predictions = model.predict(generator, verbose=1)
    y_pred = np.argmax(predictions, axis=1)
    y_true = generator.classes

    # Compute confusion matrix
    cm = confusion_matrix(y_true, y_pred)

    # Plot
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted Label', fontsize=12, fontweight='bold')
    plt.ylabel('True Label', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix - {model_name}', fontsize=14, fontweight='bold')
    plt.xticks(rotation=45, ha='right')
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.savefig(f'./confusion_matrix_{model_name.lower().replace(" ", "_")}.png',
                dpi=300, bbox_inches='tight')
    plt.show()

    return cm, y_true, y_pred

# Use the best model (typically EfficientNet or VGG16 performs best)
cm, y_true, y_pred = plot_confusion_matrix(model_effnet, test_generator, "EfficientNetB0")


In [ ]:
def print_classification_report(y_true, y_pred, model_name="Model"):
    """Print detailed classification report"""
    print("\n" + "="*80)
    print(f"CLASSIFICATION REPORT - {model_name}")
    print("="*80 + "\n")

    report = classification_report(y_true, y_pred,
                                   target_names=CLASS_NAMES,
                                   digits=4)
    print(report)

    # Save to file
    with open(f'./classification_report_{model_name.lower().replace(" ", "_")}.txt', 'w') as f:
        f.write(f"CLASSIFICATION REPORT - {model_name}\n")
        f.write("="*80 + "\n\n")
        f.write(report)

print_classification_report(y_true, y_pred, "EfficientNetB0")

In [ ]:
def visualize_predictions(model, generator, num_images=12, correct=True):
    """Visualize model predictions"""
    generator.reset()

    # Get predictions
    images_batch, labels_batch = next(generator)
    predictions = model.predict(images_batch, verbose=0)

    # Get indices
    pred_classes = np.argmax(predictions, axis=1)
    true_classes = np.argmax(labels_batch, axis=1)

    if correct:
        indices = np.where(pred_classes == true_classes)[0]
        title = "Correct Predictions"
        filename = "correct_predictions.png"
    else:
        indices = np.where(pred_classes != true_classes)[0]
        title = "Incorrect Predictions"
        filename = "incorrect_predictions.png"

    if len(indices) == 0:
        print(f"No {title.lower()} found in this batch!")
        return

    # Limit to num_images
    indices = indices[:min(num_images, len(indices))]

    # Plot
    rows = (len(indices) + 3) // 4
    cols = min(4, len(indices))

    fig, axes = plt.subplots(rows, cols, figsize=(15, 4*rows))
    if rows == 1:
        axes = axes.reshape(1, -1)

    for idx, ax_idx in enumerate(indices):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]

        img = images_batch[ax_idx]
        true_class = CLASS_NAMES[true_classes[ax_idx]]
        pred_class = CLASS_NAMES[pred_classes[ax_idx]]
        confidence = predictions[ax_idx][pred_classes[ax_idx]] * 100

        ax.imshow(img)
        ax.axis('off')

        if correct:
            color = 'green'
            ax.set_title(f'True: {true_class}\nPred: {pred_class}\nConf: {confidence:.1f}%',
                        fontsize=9, color=color, fontweight='bold')
        else:
            color = 'red'
            ax.set_title(f'True: {true_class}\nPred: {pred_class}\nConf: {confidence:.1f}%',
                        fontsize=9, color=color, fontweight='bold')

    # Hide unused subplots
    for idx in range(len(indices), rows * cols):
        row = idx // cols
        col = idx % cols
        ax = axes[row, col] if rows > 1 else axes[col]
        ax.axis('off')

    plt.suptitle(title, fontsize=16, fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'./{filename}', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize correct predictions
visualize_predictions(model_effnet, test_generator, num_images=12, correct=True)